## 3.15 Random Values: completed work

**Adhvay Iyer · RazorCrest00 · September 29, 2026**

[Assignment](https://pages.opencodingsociety.com/python/random/2027) · [Download executed notebook](https://raw.githubusercontent.com/RazorCrest00/portfolio/main/_notebooks/homework/2026-09-29-random-hw.ipynb)

**Self-assessment: 1.00 / 1.00.** The evidence below maps completed work to the assignment; this is not a teacher-issued grade. Examples and explanations were prepared with AI assistance. Popcorn solutions are recorded here without claiming unverified classroom participation.

Each Python cell runs independently. Saved output is included in the notebook; **Run** produces fresh output in this page. The examples use classroom data, not live backend records.


## Popcorn 1: Valid random record IDs

**Prediction:** Every ID is an integer in `{1, 2, 3, 4}` because `randint(1, record_count)` includes both endpoints. Repeats are allowed: choosing one ID does not remove it. A fixed local seed makes saved evidence reproducible; change the seed to explore another sequence. A short sample is not proof that all outcomes are equally likely.


In [1]:
# CODE_RUNNER: 3.15 Popcorn 1 - Inclusive record IDs
import random

record_count = 4
rng = random.Random(315)
selected_ids = [rng.randint(1, record_count) for _ in range(12)]
print("Possible IDs:", list(range(1, record_count + 1)))
print("Selected IDs:", selected_ids)
assert all(type(value) is int and 1 <= value <= record_count for value in selected_ids)
print("Every ID is in range: PASS")
print("Repeated choices are allowed; selection does not remove a record.")


Possible IDs: [1, 2, 3, 4]
Selected IDs: [4, 4, 3, 1, 3, 1, 1, 2, 3, 2, 3, 1]
Every ID is in range: PASS
Repeated choices are allowed; selection does not remove a record.


## Popcorn 2: Select a complete structured record

Choose the dictionary once, then read all its fields. Choosing the product, category, and spec separately could assemble a record that never existed. **Prediction:** Every displayed triple must exactly match one of the three provided records.


In [2]:
# CODE_RUNNER: 3.15 Popcorn 2 - Keep record fields together
import random

parts = [
    {"product_name": "Replacement Flywheels", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]

rng = random.Random(316)
for trial in range(1, 5):
    selected = rng.choice(parts)
    print(trial, selected["product_name"], "|", selected["category"], "|", selected["spec_number"])
    assert selected in parts


1 Replacement Flywheels | Auto Racing | 1.1
2 Replacement Flywheels | Auto Racing | 1.1
3 Multiple Disc Clutch Assemblies | Drag Racing | 1.2
4 Replacement Flywheels | Auto Racing | 1.1


## Popcorn 3: Random choices drive the response

Both the part and the action are chosen. The selected spec is looked up before deciding the response: searching for a missing record returns 404, creating an existing spec returns 409, and updating requires an existing record. The set represents a small classroom backend simulation; no HTTP request is sent.


In [3]:
# CODE_RUNNER: 3.15 Popcorn 3 - Random API action
import random

parts = [
    {"product_name": "Replacement Flywheels", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]

def apply_action(part, action, existing_specs):
    """Simulate an action against an in-memory set; return status and explanation."""
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in existing_specs:
            return 200, "Record found"
        return 404, "Record not found"
    elif action == "POST/create":
        if spec in existing_specs:
            return 409, "Duplicate spec rejected"
        existing_specs.add(spec)
        return 201, "Record created"
    elif action == "PUT/update":
        if spec in existing_specs:
            return 200, "Existing record update accepted"
        return 404, "Cannot update missing record"
    elif action == "DELETE/remove":
        if spec in existing_specs:
            existing_specs.remove(spec)
            return 204, "Record removed"
        return 404, "Cannot remove missing record"
    else:
        return 400, "Unknown action"

actions = ["GET/search", "POST/create", "PUT/update"]
existing_specs = {"1.1", "2.1"}
rng = random.Random(317)
for trial in range(1, 7):
    part = rng.choice(parts)
    action = rng.choice(actions)
    status, result = apply_action(part, action, existing_specs)
    print(trial, action, part["spec_number"], "->", status, result)


1 GET/search 1.2 -> 404 Record not found
2 POST/create 2.1 -> 409 Duplicate spec rejected
3 GET/search 1.1 -> 200 Record found
4 PUT/update 1.1 -> 200 Existing record update accepted
5 PUT/update 1.2 -> 404 Cannot update missing record
6 POST/create 1.2 -> 201 Record created


## Homework: Reusable SFI backend QA simulator

`simulate_qa` accepts the record list, allowed actions, initial specs, run count, and seed. It selects a whole record and one action per run, keeps simulated state between runs, and returns a transcript plus final state. It copies the initial specs so repeated calls start fresh without altering caller data.

The response depends on **current** state: a successful create makes a later create a duplicate; a delete makes a later search missing. `PUT/update` models accepting a change to an existing record; it does not invent new field values. The four actions and three parts give 12 possible part/action combinations before considering state. No finite random run guarantees every combination.


In [4]:
# CODE_RUNNER: 3.15 Homework - Randomized QA simulator
import random


def apply_action(part, action, existing_specs):
    """Simulate an action against an in-memory set; return status and explanation."""
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in existing_specs:
            return 200, "Record found"
        return 404, "Record not found"
    elif action == "POST/create":
        if spec in existing_specs:
            return 409, "Duplicate spec rejected"
        existing_specs.add(spec)
        return 201, "Record created"
    elif action == "PUT/update":
        if spec in existing_specs:
            return 200, "Existing record update accepted"
        return 404, "Cannot update missing record"
    elif action == "DELETE/remove":
        if spec in existing_specs:
            existing_specs.remove(spec)
            return 204, "Record removed"
        return 404, "Cannot remove missing record"
    else:
        return 400, "Unknown action"

def simulate_qa(parts, actions, existing_spec_numbers, test_count, seed):
    if type(test_count) is not int or test_count < 0:
        raise ValueError("test_count must be a nonnegative whole number")
    if test_count and (not parts or not actions):
        raise ValueError("Nonempty records and actions are required for positive runs")
    rng = random.Random(seed)
    existing = set(existing_spec_numbers)
    transcript = []
    for trial in range(1, test_count + 1):
        part = rng.choice(parts)
        action = rng.choice(actions)
        status, message = apply_action(part, action, existing)
        transcript.append({"trial": trial, "product_name": part["product_name"],
                           "category": part["category"], "spec_number": part["spec_number"],
                           "action": action, "status": status, "message": message})
    return transcript, sorted(existing)
parts = [
    {"product_name": "Replacement Flywheels", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]

actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]
existing_spec_numbers = ["1.1", "2.1"]
test_count = 12
seed = 315
transcript, final_specs = simulate_qa(parts, actions, existing_spec_numbers, test_count, seed)
for row in transcript:
    print(str(row["trial"]) + ": " + row["action"] + " | " + row["product_name"] +
          " | " + row["category"] + " | " + row["spec_number"] +
          " -> " + str(row["status"]) + " " + row["message"])
print("Final existing specs:", final_specs)
print("Original input specs:", existing_spec_numbers)


1: DELETE/remove | Multiple Disc Clutch Assemblies | Drag Racing | 1.2 -> 404 Cannot remove missing record
2: GET/search | Multiple Disc Clutch Assemblies | Drag Racing | 1.2 -> 404 Record not found
3: GET/search | Multiple Disc Clutch Assemblies | Drag Racing | 1.2 -> 404 Record not found
4: POST/create | Replacement Flywheels | Auto Racing | 1.1 -> 409 Duplicate spec rejected
5: PUT/update | Racing Flywheel Record | Auto Racing | 2.1 -> 200 Existing record update accepted
6: PUT/update | Replacement Flywheels | Auto Racing | 1.1 -> 200 Existing record update accepted
7: GET/search | Replacement Flywheels | Auto Racing | 1.1 -> 200 Record found
8: POST/create | Replacement Flywheels | Auto Racing | 1.1 -> 409 Duplicate spec rejected
9: GET/search | Replacement Flywheels | Auto Racing | 1.1 -> 200 Record found
10: PUT/update | Racing Flywheel Record | Auto Racing | 2.1 -> 200 Existing record update accepted
11: DELETE/remove | Racing Flywheel Record | Auto Racing | 2.1 -> 204 Record re

## Tests: Randomness plus deterministic branch coverage

Random trials explore combinations; explicit action tests establish correctness even if a particular random sample misses a branch. The sequence below proves missing-search, create, duplicate-create, update, delete, and missing-delete behavior. A repeated seed must reproduce the transcript, zero runs must preserve initial state, and bad run counts must be rejected.

AP `RANDOM(1, 4)` matches Python `randint(1, 4)`. AP list positions start at 1; Python list positions start at 0, so choosing an index would require `randint(0, len(parts) - 1)`. `choice(parts)` avoids that manual conversion.


In [5]:
# CODE_RUNNER: 3.15 Tests - Stateful action paths
import random


def apply_action(part, action, existing_specs):
    """Simulate an action against an in-memory set; return status and explanation."""
    spec = part["spec_number"]
    if action == "GET/search":
        if spec in existing_specs:
            return 200, "Record found"
        return 404, "Record not found"
    elif action == "POST/create":
        if spec in existing_specs:
            return 409, "Duplicate spec rejected"
        existing_specs.add(spec)
        return 201, "Record created"
    elif action == "PUT/update":
        if spec in existing_specs:
            return 200, "Existing record update accepted"
        return 404, "Cannot update missing record"
    elif action == "DELETE/remove":
        if spec in existing_specs:
            existing_specs.remove(spec)
            return 204, "Record removed"
        return 404, "Cannot remove missing record"
    else:
        return 400, "Unknown action"

def simulate_qa(parts, actions, existing_spec_numbers, test_count, seed):
    if type(test_count) is not int or test_count < 0:
        raise ValueError("test_count must be a nonnegative whole number")
    if test_count and (not parts or not actions):
        raise ValueError("Nonempty records and actions are required for positive runs")
    rng = random.Random(seed)
    existing = set(existing_spec_numbers)
    transcript = []
    for trial in range(1, test_count + 1):
        part = rng.choice(parts)
        action = rng.choice(actions)
        status, message = apply_action(part, action, existing)
        transcript.append({"trial": trial, "product_name": part["product_name"],
                           "category": part["category"], "spec_number": part["spec_number"],
                           "action": action, "status": status, "message": message})
    return transcript, sorted(existing)
parts = [
    {"product_name": "Replacement Flywheels", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
]

part = parts[1]
existing = {"1.1", "2.1"}
sequence = [("GET/search", 404), ("POST/create", 201), ("POST/create", 409),
            ("PUT/update", 200), ("GET/search", 200), ("DELETE/remove", 204),
            ("DELETE/remove", 404), ("PUT/update", 404), ("UNKNOWN", 400)]
for action, expected in sequence:
    status, message = apply_action(part, action, existing)
    assert status == expected
    print(action, "->", status, message)
assert existing == {"1.1", "2.1"}
actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]
initial = ["1.1", "2.1"]
first = simulate_qa(parts, actions, initial, 12, 315)
assert first == simulate_qa(parts, actions, initial, 12, 315)
assert initial == ["1.1", "2.1"]
assert simulate_qa([], [], initial, 0, 315) == ([], ["1.1", "2.1"])
for invalid in [-1, 1.5, True]:
    try:
        simulate_qa(parts, actions, initial, invalid, 315)
    except ValueError:
        print("Invalid run count rejected:", invalid)
    else:
        raise AssertionError("Invalid run count was accepted")
print("State transitions, repeatable seed, zero runs, and unchanged input: PASS")


GET/search -> 404 Record not found
POST/create -> 201 Record created
POST/create -> 409 Duplicate spec rejected
PUT/update -> 200 Existing record update accepted
GET/search -> 200 Record found
DELETE/remove -> 204 Record removed
DELETE/remove -> 404 Cannot remove missing record
PUT/update -> 404 Cannot update missing record
UNKNOWN -> 400 Unknown action
Invalid run count rejected: -1
Invalid run count rejected: 1.5
Invalid run count rejected: True
State transitions, repeatable seed, zero runs, and unchanged input: PASS


## MCQ: 4 / 4

`MCQ 3.15: 4/4 | answers: C,B,B,B`

| Question | Answer and reason |
| --- | --- |
| `randint(2, 5)` range | C: 2, 3, 4, 5; both endpoints included. |
| Same record twice | B: repeats are allowed. |
| AP `RANDOM(1,4)` equivalent | B: `random.randint(1,4)`. |
| Why one run is insufficient | B: it shows only one possible outcome. |

The answers were checked in the lesson's client-side quiz with AI assistance. A fixed seed provides reproducibility, not evidence of fairness or cryptographic security.


## Rubric evidence

| Criterion | Self-score | Evidence |
| --- | ---: | --- |
| Popcorn 1 | 0.10 / 0.10 | Inclusive IDs, repeats and possible range explained |
| Popcorn 2 | 0.15 / 0.15 | Whole structured records selected and displayed |
| Popcorn 3 | 0.20 / 0.20 | Random record/action and action-dependent result |
| MCQ | 0.15 / 0.15 | 4/4 result and explanations |
| Homework simulator | 0.40 / 0.40 | Reusable 12-run simulation; stateful duplicate handling; explicit branch tests |


## Design thinking and browser execution

**Need:** Make the decision or calculation clear enough to inspect. **Goal:** Correct results for ordinary and boundary inputs. **Choice:** Keep each task self-contained and give important decisions descriptive names. **Prototype:** The completed cells above. **Test and revision:** Compare expected outputs with execution, then check changed inputs and boundary cases; task-specific findings are recorded above.

The page uses the portfolio's existing OCS editor and Pyodide runtime to execute Python in the browser. This makes the submitted work editable and reproducible beyond static code. OCS/Pyodide provide that infrastructure; no new CSS or shared runner changes are claimed. An initial runtime download is required. These are code checks, not a claim of a peer playtest.


## Submission notes

**Page:** https://razorcrest00.github.io/portfolio/python/random-hw

[Download the self-contained homework code](https://razorcrest00.github.io/portfolio/assets/homework/2026-09-29/random-values.py). Where the [lesson](https://pages.opencodingsociety.com/python/random/2027) has a Code Submission form, choose Python and paste the file contents without Markdown fences. Include this page link and the following notes. The form has not been submitted automatically.

```text
Lesson: 3.15 Random Values
Portfolio: https://razorcrest00.github.io/portfolio/python/random-hw
MCQ 3.15: 4/4 | answers: C,B,B,B
Homework: 12 random trials with seed 315; all action/status paths also checked deterministically
Popcorn work: completed and tested in this notebook
Tests: see saved outputs and verification evidence
Self-assessment: see criterion-by-criterion table
```

**Sources:** [Current assignment](https://pages.opencodingsociety.com/python/random/2027), checked September 29, 2026; [Python language reference](https://docs.python.org/3/reference/); [College Board pseudocode reference](https://apcentral.collegeboard.org/media/pdf/ap-computer-science-principles-exam-reference-sheet.pdf). Attribution includes the assignment's supplied data and starter structure.

<script>
document.addEventListener('DOMContentLoaded', () => {
  document.querySelectorAll('.code-runner-container .languageSelect').forEach(select => {
    select.replaceChildren(new Option('Python', 'python'));
  });
});
</script>
